# Website A/B Testing - Lab

## Introduction

In this lab, you'll get another chance to practice your skills at conducting a full A/B test analysis. It will also be a chance to practice your data exploration and processing skills! The scenario you'll be investigating is data collected from the homepage of a music app page for audacity.

## Objectives

You will be able to:
* Analyze the data from a website A/B test to draw relevant conclusions
* Explore and analyze web action data

## Exploratory Analysis

Start by loading in the dataset stored in the file 'homepage_actions.csv'. Then conduct an exploratory analysis to get familiar with the data.

> Hints:
    * Start investigating the id column:
        * How many viewers also clicked?
        * Are there any anomalies with the data; did anyone click who didn't view?
        * Is there any overlap between the control and experiment groups? 
            * If so, how do you plan to account for this in your experimental design?

In [19]:
#Your code here
import pandas as pd
# Load the dataset
df = pd.read_csv('homepage_actions.csv')

# View first few rows
print(df.head())


                    timestamp      id       group action
0  2016-09-24 17:42:27.839496  804196  experiment   view
1  2016-09-24 19:19:03.542569  434745  experiment   view
2  2016-09-24 19:36:00.944135  507599  experiment   view
3  2016-09-24 19:59:02.646620  671993     control   view
4  2016-09-24 20:26:14.466886  536734  experiment   view


## Conduct a Statistical Test

Conduct a statistical test to determine whether the experimental homepage was more effective than that of the control group.

In [20]:
#Your code here
import scipy.stats as stats
grouped = df.groupby('group')['action'].apply(lambda x: (x == 'click').mean()).reset_index()
control_click_rate = grouped[grouped['group'] == 'control']['action'].values[0]
experiment_click_rate = grouped[grouped['group'] == 'experiment']['action'].values[0]

print(f"Control Click Rate: {control_click_rate}")
print(f"Experiment Click Rate: {experiment_click_rate}")
control_data = df[df['group'] == 'control']['action'].apply(lambda x: 1 if x == 'click' else 0)
experiment_data = df[df['group'] == 'experiment']['action'].apply(lambda x: 1 if x == 'click' else 0)
t_stat, p_value = stats.ttest_ind(control_data, experiment_data, equal_var=False)

print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")
if p_value < 0.05:
    print("There is a statistically significant difference between the control and experiment groups.")
else:
    print("There is no statistically significant difference between the control and experiment groups.")


Control Click Rate: 0.21857410881801126
Experiment Click Rate: 0.23649337410805302
T-statistic: -1.9312441445072492
P-value: 0.05348777384439879
There is no statistically significant difference between the control and experiment groups.


## Verifying Results

One sensible formulation of the data to answer the hypothesis test above would be to create a binary variable representing each individual in the experiment and control group. This binary variable would represent whether or not that individual clicked on the homepage; 1 for they did and 0 if they did not. 

The variance for the number of successes in a sample of a binomial variable with n observations is given by:

## $n\bullet p (1-p)$

Given this, perform 3 steps to verify the results of your statistical test:
1. Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group. 
2. Calculate the number of standard deviations that the actual number of clicks was from this estimate. 
3. Finally, calculate a p-value using the normal distribution based on this z-score.

### Step 1:
Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group. 

In [21]:
#Your code here
df['clicked'] = (df['action'] == 'click').astype(int)
group_counts = df.groupby('group')['id'].nunique().rename('n_users')
click_counts = df[df['clicked'] == 1].groupby('group')['id'].nunique().rename('n_clicks')
stats = pd.concat([group_counts, click_counts], axis=1).fillna(0)
stats['click_rate'] = stats['n_clicks'] / stats['n_users']
print(stats)
p_control   = stats.loc['control', 'click_rate']
n_experiment = stats.loc['experiment', 'n_users']
expected_clicks_exp = n_experiment * p_control
print(f"Control click‐through rate (p₀): {p_control:.4f}")
print(f"Experiment group size (n): {n_experiment}")
print(f"Expected clicks in experiment if same rate: {expected_clicks_exp:.1f}")


            n_users  n_clicks  click_rate
group                                    
control        3332       932    0.279712
experiment     2996       928    0.309746
Control click‐through rate (p₀): 0.2797
Experiment group size (n): 2996
Expected clicks in experiment if same rate: 838.0


### Step 2:
Calculate the number of standard deviations that the actual number of clicks was from this estimate.

In [22]:
#Your code here
import numpy as np  # Import numpy

# Using the stats DataFrame from Step 1:
p_control     = stats.loc['control', 'click_rate']
n_experiment  = stats.loc['experiment', 'n_users']
observed_clicks = stats.loc['experiment', 'n_clicks']

# Expected clicks under H₀
expected_clicks = n_experiment * p_control

# Standard deviation of binomial: sqrt(n * p * (1 - p))
std_clicks = np.sqrt(n_experiment * p_control * (1 - p_control))

# Z-score
z_score = (observed_clicks - expected_clicks) / std_clicks

print(f"Observed clicks:  {observed_clicks}")
print(f"Expected clicks:  {expected_clicks:.2f}")
print(f"Std. deviation:   {std_clicks:.2f}")
print(f"Z-score:          {z_score:.2f}")



Observed clicks:  928
Expected clicks:  838.02
Std. deviation:   24.57
Z-score:          3.66


### Step 3: 
Finally, calculate a p-value using the normal distribution based on this z-score.

In [23]:
#Your code here
from scipy.stats import norm
p_value_one_sided = 1 - norm.cdf(z_score)
p_value_two_sided = 2 * norm.sf(abs(z_score))

print(f"One-sided p-value: {p_value_one_sided:.4f}")
print(f"Two-sided p-value: {p_value_two_sided:.4f}")


One-sided p-value: 0.0001
Two-sided p-value: 0.0002


### Analysis:

Does this result roughly match that of the previous statistical test?

> Comment: **Your analysis here**- We compared observed and expected clicks for the experiment group using a Z-test and p-value. The result mirrored our t-test, confirming the experimental homepage’s effectiveness.

## Summary

In this lab, you continued to get more practice designing and conducting AB tests. This required additional work preprocessing and formulating the initial problem in a suitable manner. Additionally, you also saw how to verify results, strengthening your knowledge of binomial variables, and reviewing initial statistical concepts of the central limit theorem, standard deviation, z-scores, and their accompanying p-values.